#Day71 - JSON

In [0]:
dfjson1 = spark.read.json("/Volumes/workspace/default/volumewd36/simple_json.txt")
dfjson1.printSchema()
display(dfjson1)

In [0]:
# primitiveAsString - > don't do schema inference
dfjson2 = spark.read.json("/Volumes/workspace/default/volumewd36/simple_json.txt", primitivesAsString=True)
dfjson2.printSchema()
display(dfjson2)

In [0]:
str1 = 'id int, name string, amt double, dop string'
dfjson3 = spark.read.schema(str1).json("/Volumes/workspace/default/volumewd36/simple_json.txt")
dfjson3.printSchema()
display(dfjson3)

In [0]:
dfjson4 = spark.read.json("/Volumes/workspace/default/volumewd36/simple_json.txt", prefersDecimal=True)
dfjson4.printSchema()
display(dfjson4)

In [0]:
dfjson5 = spark.read.json("/Volumes/workspace/default/volumewd36/simple_json2.txt")
dfjson5.printSchema()
display(dfjson1)

In [0]:
str1 = 'id int, name string, amt double, dop string,corrupt_record string'
dfjson6 = spark.read.schema(str1).json("/Volumes/workspace/default/volumewd36/simple_json2.txt",allowComments=True )
dfjson6.printSchema()
display(dfjson3)

# merge schema

### case study
#### Q: if the source (external) is sending data to us, if our customer (data scientist / data analyst) is directly communincated with our source system  and asked them to propagate more/less attributes/featurs without the knowlegde of  the data engineering team. How to handle this?
#### A: Here we have to implement the strategy of schema evolution using merge schema
##### steps to follow:
- 1. Collect the data as it is from the source
- 2. Convert it int orc/parquet format and write to the target by appending the data on a regular interval
- 3. Read the data from the target and do teh schema evolution and get the evolved dataframe created

In [0]:
# /Volumes/workspace/default/volumewd36/mergeSchemaDay1.txt
# /Volumes/workspace/default/volumewd36/mergeSchemaDay2.txt
# /Volumes/workspace/default/volumewd36/mergeSchemaDay3.txt

# Day1
## collect the data as it is by using infer schema
day1_df = spark.read.csv("/Volumes/workspace/default/volumewd36/mergeSchemaDay1.txt", header=True, inferSchema=True)
## write to orc / serialized format
day1_df.write.orc("/Volumes/workspace/wd36schema/ingestion_volume/target/orcout_merged",mode='overwrite')
#day1_df.printSchema()
#display(day1_df)

post_day1 = spark.read.orc("/Volumes/workspace/wd36schema/ingestion_volume/target/orcout_merged")
post_day1.printSchema()
display(post_day1)


In [0]:
# Day2
## collect the data as it is by using infer schema
day2_df = spark.read.csv("/Volumes/workspace/default/volumewd36/mergeSchemaDay2.txt", header=True, inferSchema=True)
## write to orc / serialized format
day2_df.write.orc("/Volumes/workspace/wd36schema/ingestion_volume/target/orcout_merged",mode='append')
day2_df.printSchema()
display(day2_df)

post_day2 = spark.read.orc("/Volumes/workspace/wd36schema/ingestion_volume/target/orcout_merged")
post_day2.printSchema()
display(post_day2)

In [0]:
# Day3
## collect the data as it is by using infer schema
day3_df = spark.read.csv("/Volumes/workspace/default/volumewd36/mergeSchemaDay3.txt", header=True, inferSchema=True)
## write to orc / serialized format
day3_df.write.orc("/Volumes/workspace/wd36schema/ingestion_volume/target/orcout_merged",mode='append')
day3_df.printSchema()
display(day3_df)

post_day3 = spark.read.orc("/Volumes/workspace/wd36schema/ingestion_volume/target/orcout_merged")
post_day3.printSchema()
display(post_day3)

In [0]:
post_day3 = spark.read.orc("/Volumes/workspace/wd36schema/ingestion_volume/target/orcout_merged",mergeSchema=True)
post_day3.printSchema()
display(post_day3)

# advance write

In [0]:
strct1 = 'custid int, name string, age int, corrupted_record string'
input_df = spark.read.schema(strct1).csv("/Volumes/workspace/default/volumewd36/malformed_data_with_spl_characters.txt", header=False,sep=',',mode='permissive', columnNameOfCorruptRecord='corrupted_record', quote="'", escape="\\")
input_df.show()

In [0]:
# importan write options 
# path, mode, compress, sep, quote, escape, header
input_df.repartition(4).write.csv(path='/Volumes/workspace/default/volumewd36/csv_adv_write', mode='overwrite', compression='None', sep='|', quote="'", escape='~', header=True)



# JSON write options - Advanced
### **very important** path, mode, compression
#### **importan** 

In [0]:
#dfjson6.write.json(path="/Volumes/workspace/default/volumewd36/json_adv_write",mode="append",compression='snappy')
dfjson6.write.json(path="/Volumes/workspace/default/volumewd36/json_adv_write",mode="append",compression='None')

In [0]:
%sh
ls -l /Volumes/workspace/default/volumewd36/json_adv_write

In [0]:
input_df.write.orc(path="/Volumes/workspace/default/volumewd36/orc_adv_out",mode="overwrite",partitionBy='age')

In [0]:
%sh
ls -l /Volumes/workspace/wd36schema/ingestion_volume/target/orc_adv_out

In [0]:
spark.read.orc("/Volumes/workspace/default/volumewd36/orc_adv_out").show()

In [0]:
spark.read.orc("/Volumes/workspace/default/volumewd36/orc_adv_out").where('age=30').show()
# PUSH DOWN OPTIMIZATION / PARTITION PRUNING